# BPIC-17 Process Discovery and Validation

## 1. Title and Objective

This notebook performs the second stage of the assignment: discovering process models from the BPIC-17 event log and validating their quality with conformance-oriented metrics.

The notebook first inspects lifecycle information, then uses a complete-event representation of the log for process discovery. It compares Inductive Miner, Heuristics Miner, Alpha Miner, and a directly-follows graph (DFG). The output is intended as intermediate material for the report section on process model creation and validation, not as final report text.


## 2. Imports and Configuration

Paths are resolved from the repository root. The raw XES file remains local and ignored by Git. All generated artifacts are written to `results/` and `figures/`.


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import math
import traceback
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pm4py


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root without relying on absolute paths."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() or (candidate / "requirements.txt").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "BPI Challenge 2017.xes"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
MODEL_RESULTS_DIR = RESULTS_DIR / "process_models"
MODEL_FIGURES_DIR = FIGURES_DIR / "process_models"

for directory in [RESULTS_DIR, FIGURES_DIR, MODEL_RESULTS_DIR, MODEL_FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CASE_ID_COL = "case:concept:name"
ACTIVITY_COL = "concept:name"
TIMESTAMP_COL = "time:timestamp"
LIFECYCLE_COL = "lifecycle:transition"
EVENT_ORIGIN_COL = "EventOrigin"
COMPLETE_VALUE = "complete"

RANDOM_SEED = 42
SAMPLE_CASES = 5000
DFG_MAX_EDGES = 80
INDUCTIVE_NOISE_THRESHOLD = 0.0
HEURISTICS_DEPENDENCY_THRESHOLD = 0.5
HEURISTICS_AND_THRESHOLD = 0.65
HEURISTICS_LOOP_TWO_THRESHOLD = 0.5

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

print(f"PM4Py version: {getattr(pm4py, '__version__', 'unknown')}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")


## 3. Load Event Log

The BPIC-17 XES file is loaded with PM4Py and converted to a pandas DataFrame. The notebook keeps the original lifecycle information for inspection before any filtering.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "BPIC-17 event log not found. Place the file at "
        f"{DATA_PATH.relative_to(PROJECT_ROOT)}"
    )

log = pm4py.read_xes(str(DATA_PATH))
df = pm4py.convert_to_dataframe(log)

if TIMESTAMP_COL in df.columns:
    df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce", utc=True)

print("Raw DataFrame shape:", df.shape)
print("Columns:", list(df.columns))
print("Dtypes:")
print(df.dtypes)

df.head(10)


## 4. Lifecycle Inspection Before Filtering

Before filtering to complete events, the notebook records the lifecycle distribution and the `EventOrigin ? lifecycle:transition` crosstab. This preserves evidence for the preprocessing choice used in process discovery.


In [ ]:
if LIFECYCLE_COL not in df.columns:
    raise KeyError(f"Required lifecycle column '{LIFECYCLE_COL}' is missing.")

lifecycle_distribution = (
    df[LIFECYCLE_COL]
    .value_counts(dropna=False)
    .rename_axis(LIFECYCLE_COL)
    .reset_index(name="event_count")
)
lifecycle_distribution["event_share"] = lifecycle_distribution["event_count"] / len(df)
lifecycle_distribution.to_csv(RESULTS_DIR / "lifecycle_transition_distribution.csv", index=False)

if EVENT_ORIGIN_COL in df.columns:
    event_origin_lifecycle_crosstab = pd.crosstab(
        df[EVENT_ORIGIN_COL], df[LIFECYCLE_COL], dropna=False
    )
else:
    warnings.warn(f"Column '{EVENT_ORIGIN_COL}' is missing; crosstab will be empty.", stacklevel=2)
    event_origin_lifecycle_crosstab = pd.DataFrame()

event_origin_lifecycle_crosstab.to_csv(RESULTS_DIR / "event_origin_lifecycle_crosstab.csv")

display(lifecycle_distribution)
display(event_origin_lifecycle_crosstab)


## 5. Preprocessing Strategy

The discovery input keeps only `lifecycle:transition == "complete"` events. The rationale is that complete events represent the completion of business activities and reduce noise from workflow task states such as schedule, start, suspend, and resume. This makes the discovered model closer to the business process rather than a task lifecycle model.

This choice has a risk: lifecycle states such as start, schedule, withdraw, suspend, and resume may contain operational information that is lost in the complete-event abstraction. The complete-event log is therefore used for process discovery and quality evaluation, while the lifecycle distributions above document what was filtered out.

The preprocessing does not remove A/W/O activity groups, low-frequency activities, or low-frequency variants.


In [ ]:
required_columns = [CASE_ID_COL, ACTIVITY_COL, TIMESTAMP_COL, LIFECYCLE_COL]
missing_required = [column for column in required_columns if column not in df.columns]
if missing_required:
    raise KeyError(f"Required columns missing from event log: {missing_required}")

complete_log_df = df[df[LIFECYCLE_COL] == COMPLETE_VALUE].copy()
complete_log_df = complete_log_df.sort_values([CASE_ID_COL, TIMESTAMP_COL]).reset_index(drop=True)

if complete_log_df.empty:
    raise ValueError("The complete-filtered log is empty. Check lifecycle values before discovery.")

complete_summary = {
    "raw_events": len(df),
    "complete_events": len(complete_log_df),
    "raw_cases": df[CASE_ID_COL].nunique(),
    "complete_cases": complete_log_df[CASE_ID_COL].nunique(),
    "complete_activities": complete_log_df[ACTIVITY_COL].nunique(),
    "complete_event_origins": complete_log_df[EVENT_ORIGIN_COL].nunique() if EVENT_ORIGIN_COL in complete_log_df.columns else np.nan,
}

for key, value in complete_summary.items():
    print(f"{key}: {value}")

if EVENT_ORIGIN_COL in complete_log_df.columns:
    display(complete_log_df[EVENT_ORIGIN_COL].value_counts().to_frame("complete_event_count"))


## 6. Fixed Sample for Fallback

The full complete-filtered log is the primary input for discovery and quality evaluation. A deterministic case sample is prepared only as a fallback if a miner, metric, or visualization fails on the full log.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
unique_cases = complete_log_df[CASE_ID_COL].drop_duplicates().to_numpy()
actual_sample_cases = min(SAMPLE_CASES, len(unique_cases))
sampled_cases = rng.choice(unique_cases, size=actual_sample_cases, replace=False)
sample_log_df = (
    complete_log_df[complete_log_df[CASE_ID_COL].isin(sampled_cases)]
    .sort_values([CASE_ID_COL, TIMESTAMP_COL])
    .reset_index(drop=True)
)

print(f"Fallback sample cases: {actual_sample_cases}")
print(f"Fallback sample events: {len(sample_log_df)}")


## 7. Helper Functions

These functions keep discovery, export, visualization, metric calculation, and error handling reproducible. Failures are recorded instead of interrupting the notebook.


In [ ]:
def short_error(exc: BaseException, max_chars: int = 500) -> str:
    message = f"{type(exc).__name__}: {exc}"
    return message[:max_chars]


def save_placeholder_figure(path: Path, title: str, message: str) -> None:
    """Create a placeholder figure when PM4Py/Graphviz visualization fails."""
    plt.figure(figsize=(10, 6))
    plt.axis("off")
    plt.title(title)
    wrapped = "\n".join(message[i : i + 100] for i in range(0, len(message), 100))
    plt.text(0.02, 0.5, wrapped, va="center", ha="left", fontsize=10)
    plt.tight_layout()
    plt.savefig(path, dpi=300 if path.suffix.lower() == ".png" else None)
    plt.close()


def structural_metrics(record: dict) -> dict[str, float]:
    net = record.get("net")
    if net is None:
        return {
            "number_of_places": np.nan,
            "number_of_transitions": np.nan,
            "number_of_arcs": np.nan,
            "average_arc_degree": np.nan,
            "silent_transition_ratio": np.nan,
        }

    number_of_places = len(net.places)
    number_of_transitions = len(net.transitions)
    number_of_arcs = len(net.arcs)
    denominator = number_of_places + number_of_transitions
    silent_transitions = sum(1 for transition in net.transitions if transition.label is None)

    return {
        "number_of_places": number_of_places,
        "number_of_transitions": number_of_transitions,
        "number_of_arcs": number_of_arcs,
        "average_arc_degree": number_of_arcs / denominator if denominator else np.nan,
        "silent_transition_ratio": silent_transitions / number_of_transitions if number_of_transitions else np.nan,
    }


def extract_fitness_value(result: object) -> float:
    if isinstance(result, dict):
        for key in ["log_fitness", "average_trace_fitness", "percentage_of_fitting_traces"]:
            if key in result:
                return float(result[key])
        for value in result.values():
            if isinstance(value, (int, float, np.floating)):
                return float(value)
        return np.nan
    return float(result)


def discover_with_full_then_sample(display_name: str, slug: str, discovery_function) -> dict:
    record = {
        "algorithm": display_name,
        "slug": slug,
        "status": "failed",
        "discovery_scope": "failed",
        "discovery_used_fallback": False,
        "net": None,
        "im": None,
        "fm": None,
        "process_tree": None,
        "bpmn": None,
        "error_note": "",
        "visualization_note": "",
    }

    try:
        record.update(discovery_function(complete_log_df))
        record["status"] = "success"
        record["discovery_scope"] = "complete_filtered_full_log"
        return record
    except Exception as full_exc:
        full_error = short_error(full_exc)

    try:
        record.update(discovery_function(sample_log_df))
        record["status"] = "success"
        record["discovery_scope"] = f"complete_filtered_sample_{actual_sample_cases}_cases"
        record["discovery_used_fallback"] = True
        record["error_note"] = f"Full discovery failed: {full_error}. Sample fallback succeeded."
        return record
    except Exception as sample_exc:
        record["error_note"] = (
            f"Full discovery failed: {full_error}. "
            f"Sample fallback failed: {short_error(sample_exc)}"
        )
        return record


def discover_inductive(log_df: pd.DataFrame) -> dict:
    tree = pm4py.discover_process_tree_inductive(
        log_df,
        noise_threshold=INDUCTIVE_NOISE_THRESHOLD,
        activity_key=ACTIVITY_COL,
        timestamp_key=TIMESTAMP_COL,
        case_id_key=CASE_ID_COL,
    )
    net, im, fm = pm4py.convert_to_petri_net(tree)
    bpmn = pm4py.convert_to_bpmn(tree)
    return {"process_tree": tree, "net": net, "im": im, "fm": fm, "bpmn": bpmn}


def discover_heuristics(log_df: pd.DataFrame) -> dict:
    net, im, fm = pm4py.discover_petri_net_heuristics(
        log_df,
        dependency_threshold=HEURISTICS_DEPENDENCY_THRESHOLD,
        and_threshold=HEURISTICS_AND_THRESHOLD,
        loop_two_threshold=HEURISTICS_LOOP_TWO_THRESHOLD,
        activity_key=ACTIVITY_COL,
        timestamp_key=TIMESTAMP_COL,
        case_id_key=CASE_ID_COL,
    )
    return {"net": net, "im": im, "fm": fm}


def discover_alpha(log_df: pd.DataFrame) -> dict:
    net, im, fm = pm4py.discover_petri_net_alpha(
        log_df,
        activity_key=ACTIVITY_COL,
        timestamp_key=TIMESTAMP_COL,
        case_id_key=CASE_ID_COL,
    )
    return {"net": net, "im": im, "fm": fm}


def export_petri_net(record: dict) -> None:
    if record.get("net") is None:
        return
    pm4py.write_pnml(
        record["net"], record["im"], record["fm"],
        str(MODEL_RESULTS_DIR / f"{record['slug']}.pnml"),
    )


def export_bpmn(record: dict) -> None:
    if record.get("bpmn") is None:
        return
    pm4py.write_bpmn(record["bpmn"], str(MODEL_RESULTS_DIR / f"{record['slug']}.bpmn"), auto_layout=False)


def save_petri_visual(record: dict) -> None:
    if record.get("net") is None:
        return
    notes = []
    for extension in ["pdf", "png"]:
        path = MODEL_FIGURES_DIR / f"{record['slug']}_petri_net.{extension}"
        try:
            pm4py.save_vis_petri_net(
                record["net"], record["im"], record["fm"], str(path),
                graph_title=f"{record['algorithm']} Petri Net",
                rankdir="LR",
            )
        except Exception as exc:
            note = f"{path.name}: {short_error(exc)}"
            notes.append(note)
            save_placeholder_figure(path, f"{record['algorithm']} Petri Net", note)
    if notes:
        record["visualization_note"] = "; ".join([record.get("visualization_note", ""), *notes]).strip("; ")


def save_process_tree_visual(record: dict) -> None:
    if record.get("process_tree") is None:
        return
    notes = []
    for extension in ["pdf", "png"]:
        path = MODEL_FIGURES_DIR / f"{record['slug']}_process_tree.{extension}"
        try:
            pm4py.save_vis_process_tree(
                record["process_tree"], str(path),
                graph_title=f"{record['algorithm']} Process Tree",
                rankdir="LR",
            )
        except Exception as exc:
            note = f"{path.name}: {short_error(exc)}"
            notes.append(note)
            save_placeholder_figure(path, f"{record['algorithm']} Process Tree", note)
    if notes:
        record["visualization_note"] = "; ".join([record.get("visualization_note", ""), *notes]).strip("; ")


def save_bpmn_visual(record: dict) -> None:
    if record.get("bpmn") is None:
        return
    notes = []
    for extension in ["pdf", "png"]:
        path = MODEL_FIGURES_DIR / f"{record['slug']}_bpmn.{extension}"
        try:
            pm4py.save_vis_bpmn(
                record["bpmn"], str(path),
                graph_title=f"{record['algorithm']} BPMN",
                rankdir="LR",
            )
        except Exception as exc:
            note = f"{path.name}: {short_error(exc)}"
            notes.append(note)
            save_placeholder_figure(path, f"{record['algorithm']} BPMN", note)
    if notes:
        record["visualization_note"] = "; ".join([record.get("visualization_note", ""), *notes]).strip("; ")


def evaluate_metric_with_fallback(record: dict, metric_name: str, metric_function, extractor=float) -> tuple[float, str, bool, str]:
    if record.get("net") is None:
        return np.nan, "not_evaluated", False, "No Petri net available."

    errors = []
    evaluation_inputs = [
        ("complete_filtered_full_log", complete_log_df),
        (f"complete_filtered_sample_{actual_sample_cases}_cases", sample_log_df),
    ]

    for scope, log_df in evaluation_inputs:
        try:
            result = metric_function(
                log_df,
                record["net"],
                record["im"],
                record["fm"],
                activity_key=ACTIVITY_COL,
                timestamp_key=TIMESTAMP_COL,
                case_id_key=CASE_ID_COL,
            )
            return extractor(result), scope, scope != "complete_filtered_full_log", "; ".join(errors)
        except Exception as exc:
            errors.append(f"{scope} {metric_name} failed: {short_error(exc)}")

    return np.nan, "failed", True, "; ".join(errors)


## 8. Process Discovery

The notebook first attempts discovery on the complete-filtered full log. If a miner fails, it retries on the fixed sample and records the fallback.


In [ ]:
model_records = [
    discover_with_full_then_sample("Inductive Miner", "inductive_miner", discover_inductive),
    discover_with_full_then_sample("Heuristics Miner", "heuristics_miner", discover_heuristics),
    discover_with_full_then_sample("Alpha Miner", "alpha_miner", discover_alpha),
]

discovery_status = pd.DataFrame(
    [
        {
            "algorithm": record["algorithm"],
            "status": record["status"],
            "discovery_scope": record["discovery_scope"],
            "discovery_used_fallback": record["discovery_used_fallback"],
            "error_note": record["error_note"],
        }
        for record in model_records
    ]
)

display(discovery_status)


## 9. Model Export and Visualization

Successful Petri nets are exported as PNML. The Inductive Miner process tree is additionally exported as BPMN. Visualizations are saved as PDF and PNG. If PM4Py/Graphviz rendering fails, a placeholder figure is written and the error is recorded.


In [ ]:
for record in model_records:
    if record["status"] != "success":
        continue
    try:
        export_petri_net(record)
    except Exception as exc:
        record["error_note"] = "; ".join([record["error_note"], f"PNML export failed: {short_error(exc)}"]).strip("; ")
    try:
        export_bpmn(record)
    except Exception as exc:
        record["error_note"] = "; ".join([record["error_note"], f"BPMN export failed: {short_error(exc)}"]).strip("; ")

    save_petri_visual(record)
    save_process_tree_visual(record)
    save_bpmn_visual(record)

visualization_status = pd.DataFrame(
    [
        {
            "algorithm": record["algorithm"],
            "visualization_note": record.get("visualization_note", ""),
            "error_note": record.get("error_note", ""),
        }
        for record in model_records
    ]
)

display(visualization_status)


## 10. Directly-Follows Graph

The DFG is discovered from the complete-filtered full log and visualized with a top-edge limit for readability. The top-edge limit affects only the figure, not the discovery log.


In [ ]:
dfg_note = ""
try:
    dfg, start_activities, end_activities = pm4py.discover_dfg(
        complete_log_df,
        activity_key=ACTIVITY_COL,
        timestamp_key=TIMESTAMP_COL,
        case_id_key=CASE_ID_COL,
    )
    for extension in ["pdf", "png"]:
        path = MODEL_FIGURES_DIR / f"dfg_top_edges.{extension}"
        try:
            pm4py.save_vis_dfg(
                dfg,
                start_activities,
                end_activities,
                str(path),
                max_num_edges=DFG_MAX_EDGES,
                graph_title=f"DFG Top {DFG_MAX_EDGES} Directly-Follows Edges",
                rankdir="LR",
            )
        except Exception as exc:
            note = f"{path.name}: {short_error(exc)}"
            dfg_note = "; ".join([dfg_note, note]).strip("; ")
            save_placeholder_figure(path, "DFG Top Edges", note)
    print(f"DFG edges discovered: {len(dfg)}")
except Exception as exc:
    dfg_note = f"DFG discovery failed: {short_error(exc)}"
    for extension in ["pdf", "png"]:
        save_placeholder_figure(MODEL_FIGURES_DIR / f"dfg_top_edges.{extension}", "DFG Top Edges", dfg_note)
    print(dfg_note)


## 11. Conformance and Quality Metrics

The complete-filtered full log is used first for each quality metric. If a metric fails on the full log, the notebook falls back to the deterministic case sample and records this explicitly.


In [ ]:
quality_rows = []

for record in model_records:
    structural = structural_metrics(record)
    metric_errors = []
    metric_scopes = []
    metric_fallbacks = []

    if record["status"] == "success":
        fitness, fitness_scope, fitness_fallback, fitness_error = evaluate_metric_with_fallback(
            record, "fitness", pm4py.fitness_token_based_replay, extract_fitness_value
        )
        precision, precision_scope, precision_fallback, precision_error = evaluate_metric_with_fallback(
            record, "precision", pm4py.precision_token_based_replay, float
        )
        generalization, generalization_scope, generalization_fallback, generalization_error = evaluate_metric_with_fallback(
            record, "generalization", pm4py.generalization_tbr, float
        )

        for error in [fitness_error, precision_error, generalization_error]:
            if error:
                metric_errors.append(error)
        metric_scopes.extend([fitness_scope, precision_scope, generalization_scope])
        metric_fallbacks.extend([fitness_fallback, precision_fallback, generalization_fallback])

        try:
            built_in_simplicity = float(pm4py.simplicity_petri_net(record["net"], record["im"], record["fm"]))
            simplicity_error = ""
        except Exception as exc:
            built_in_simplicity = np.nan
            simplicity_error = f"built-in simplicity failed: {short_error(exc)}"
            metric_errors.append(simplicity_error)
    else:
        fitness = precision = generalization = built_in_simplicity = np.nan
        fitness_scope = precision_scope = generalization_scope = "not_evaluated"
        metric_fallbacks = []
        metric_errors.append(record["error_note"])

    non_empty_scopes = [scope for scope in metric_scopes if scope and scope != "not_evaluated"]
    if non_empty_scopes and all(scope == "complete_filtered_full_log" for scope in non_empty_scopes):
        evaluation_scope = "complete_filtered_full_log"
    elif non_empty_scopes:
        evaluation_scope = "mixed_full_and_sample_fallback"
    else:
        evaluation_scope = "not_evaluated"

    used_fallback = bool(record.get("discovery_used_fallback")) or any(metric_fallbacks)
    notes = [record.get("error_note", ""), record.get("visualization_note", ""), *metric_errors]
    error_note = "; ".join(note for note in notes if note)

    quality_rows.append(
        {
            "algorithm": record["algorithm"],
            "status": record["status"],
            "discovery_scope": record["discovery_scope"],
            "evaluation_scope": evaluation_scope,
            "used_fallback": used_fallback,
            "fitness": fitness,
            "fitness_scope": fitness_scope,
            "precision": precision,
            "precision_scope": precision_scope,
            "generalization": generalization,
            "generalization_scope": generalization_scope,
            "built_in_simplicity": built_in_simplicity,
            **structural,
            "error_note": error_note,
        }
    )

process_model_quality = pd.DataFrame(quality_rows)
process_model_quality.to_csv(RESULTS_DIR / "process_model_quality.csv", index=False)
process_model_quality


## 12. Save Parameters and Summary

The parameter file documents the inputs and thresholds used in this stage. The Markdown summary is intermediate material for later report writing.


In [ ]:
parameters = pd.DataFrame(
    [
        {"parameter": "created_at", "value": datetime.now().isoformat(timespec="seconds")},
        {"parameter": "data_path", "value": str(DATA_PATH.relative_to(PROJECT_ROOT))},
        {"parameter": "pm4py_version", "value": getattr(pm4py, "__version__", "unknown")},
        {"parameter": "case_id_key", "value": CASE_ID_COL},
        {"parameter": "activity_key", "value": ACTIVITY_COL},
        {"parameter": "timestamp_key", "value": TIMESTAMP_COL},
        {"parameter": "lifecycle_key", "value": LIFECYCLE_COL},
        {"parameter": "lifecycle_filter", "value": f"{LIFECYCLE_COL} == {COMPLETE_VALUE}"},
        {"parameter": "raw_events", "value": len(df)},
        {"parameter": "complete_events", "value": len(complete_log_df)},
        {"parameter": "raw_cases", "value": df[CASE_ID_COL].nunique()},
        {"parameter": "complete_cases", "value": complete_log_df[CASE_ID_COL].nunique()},
        {"parameter": "complete_activities", "value": complete_log_df[ACTIVITY_COL].nunique()},
        {"parameter": "random_seed", "value": RANDOM_SEED},
        {"parameter": "fallback_sample_cases", "value": actual_sample_cases},
        {"parameter": "fallback_sample_events", "value": len(sample_log_df)},
        {"parameter": "dfg_max_edges", "value": DFG_MAX_EDGES},
        {"parameter": "inductive_noise_threshold", "value": INDUCTIVE_NOISE_THRESHOLD},
        {"parameter": "heuristics_dependency_threshold", "value": HEURISTICS_DEPENDENCY_THRESHOLD},
        {"parameter": "heuristics_and_threshold", "value": HEURISTICS_AND_THRESHOLD},
        {"parameter": "heuristics_loop_two_threshold", "value": HEURISTICS_LOOP_TWO_THRESHOLD},
        {"parameter": "full_log_evaluation_preferred", "value": True},
        {"parameter": "fallback_policy", "value": "Use sample only if discovery, metric, or visualization fails on full log."},
    ]
)
parameters.to_csv(RESULTS_DIR / "process_discovery_parameters.csv", index=False)

summary_lines = []
summary_lines.append("# Process Discovery and Validation Summary")
summary_lines.append("")
summary_lines.append("## Scope")
summary_lines.append("")
summary_lines.append(
    "This intermediate summary documents process discovery and model validation for the BPIC-17 event log. "
    "It compares Inductive Miner, Heuristics Miner, Alpha Miner, and a directly-follows graph using a complete-event representation of the log."
)
summary_lines.append("")
summary_lines.append("## Preprocessing")
summary_lines.append("")
summary_lines.append(
    f"The raw log contains {len(df):,} events and {df[CASE_ID_COL].nunique():,} cases. "
    f"After filtering to `{LIFECYCLE_COL} == {COMPLETE_VALUE}`, the discovery log contains "
    f"{len(complete_log_df):,} events and {complete_log_df[CASE_ID_COL].nunique():,} cases."
)
summary_lines.append("")
summary_lines.append(
    "Only complete events are used because they represent the completion of business activities and reduce noise from task lifecycle states such as schedule, start, suspend, and resume. "
    "The risk is that start, schedule, withdraw, suspend, and resume information may be lost. The original lifecycle distribution and EventOrigin-lifecycle crosstab are therefore saved separately."
)
summary_lines.append("")
summary_lines.append("## Quality Metrics")
summary_lines.append("")
def dataframe_to_markdown_table(table: pd.DataFrame) -> str:
    """Create a pipe-style Markdown table without optional dependencies."""
    display_table = table.copy()
    for column in display_table.columns:
        display_table[column] = display_table[column].map(
            lambda value: "" if pd.isna(value) else str(value).replace("|", "\\|").replace("\n", " ")
        )
    header = "| " + " | ".join(display_table.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(display_table.columns)) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in display_table.to_numpy(dtype=str)]
    return "\n".join([header, separator, *rows])

summary_lines.append(dataframe_to_markdown_table(process_model_quality))
summary_lines.append("")
summary_lines.append("## Fallback Notes")
summary_lines.append("")
if process_model_quality["used_fallback"].any():
    for _, row in process_model_quality[process_model_quality["used_fallback"]].iterrows():
        summary_lines.append(f"- {row['algorithm']}: fallback was used. {row['error_note']}")
else:
    summary_lines.append("- No discovery or metric fallback was needed; all evaluated metrics used the complete-filtered full log.")
if dfg_note:
    summary_lines.append(f"- DFG note: {dfg_note}")
summary_lines.append("")
summary_lines.append("## Generated Files")
summary_lines.append("")
for file_name in [
    "results/lifecycle_transition_distribution.csv",
    "results/event_origin_lifecycle_crosstab.csv",
    "results/process_model_quality.csv",
    "results/process_discovery_parameters.csv",
    "results/process_discovery_summary.md",
    "results/process_models/inductive_miner.pnml",
    "results/process_models/heuristics_miner.pnml",
    "results/process_models/alpha_miner.pnml",
    "results/process_models/inductive_miner.bpmn",
    "figures/process_models/inductive_miner_petri_net.pdf",
    "figures/process_models/inductive_miner_process_tree.pdf",
    "figures/process_models/inductive_miner_bpmn.pdf",
    "figures/process_models/heuristics_miner_petri_net.pdf",
    "figures/process_models/alpha_miner_petri_net.pdf",
    "figures/process_models/dfg_top_edges.pdf",
]:
    summary_lines.append(f"- `{file_name}`")
summary_lines.append("")
summary_lines.append("## Interpretation Notes for Report")
summary_lines.append("")
summary_lines.append("- Inductive Miner is the primary structured model candidate.")
summary_lines.append("- Heuristics Miner provides a frequency-aware comparison model for noisy behavior.")
summary_lines.append("- Alpha Miner is treated as a baseline and may perform worse on real-life noisy logs.")
summary_lines.append("- DFG is useful for explaining high-frequency directly-follows relations, but it is not a full behavioral model for conformance checking.")
summary_lines.append("- Fitness, precision, generalization, and simplicity should be interpreted jointly rather than as a single ranking.")

summary_path = RESULTS_DIR / "process_discovery_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

parameters


## 13. Summary for Report

Key points for the later report:

- The model discovery stage is based on complete events after documenting the original lifecycle distribution.
- Inductive Miner is used as the main structured candidate model.
- Heuristics Miner provides a noise-aware comparison model.
- Alpha Miner is included only as a baseline because real-life BPIC-17 behavior can violate its assumptions.
- DFG visualization supports intuitive discussion of high-frequency activity relations.
- Model quality is assessed with fitness, precision, generalization, built-in simplicity, and structural simplicity metrics.
- Any fallback to the deterministic sample is recorded explicitly in `process_model_quality.csv` and `process_discovery_summary.md`.

This is intermediate analysis material, not the final LaTeX report.
